In [1]:
'''
Docstring for 2501291219-05.ipynb
This notebook integrate between pipeline-01 and BigQuery input
'''

'\nDocstring for 2501291219-05.ipynb\nThis notebook integrate between pipeline-01 and BigQuery input\n'

### Import important library

In [2]:
from __future__ import annotations

import json
import os
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import yaml
from pathlib import Path
import time
from collections import defaultdict

In [3]:
from functions.utils.logging import get_logger
from functions.utils.config  import PROJECT_ROOT, load_config
from functions.utils.llm_client import build_llm_client_from_yaml
from functions.utils.text_embeddings import GoogleEmbeddingModel
from functions.core.context_builder import build_user_context
from functions.core.history import build_history_summary

### QueryData

In [4]:
from google.cloud import bigquery

In [5]:
class DataQuery:
    def __init__(self):
        self.client = bigquery.Client()
    def get_students(self):
        query = """
            SELECT *
            FROM `poc-piloturl-nonprod.gold_layer.students`
        """
        df = self.client.query(query).to_dataframe()
        return df
    def get_interactions(self):
        query = """
            SELECT *
            FROM `poc-piloturl-nonprod.gold_layer.interactions`
        """
        df = self.client.query(query).to_dataframe()
        return df 
    def get_user_events_json(self):
        query = """
        SELECT *
        FROM `poc-piloturl-nonprod.gold_layer.feeds`
        """
        df = self.client.query(query).to_dataframe()
        # ensure created_at is ISO-8601 Z format
        df["created_at"] = df["created_at"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")

        feeds_lookup: Dict[str, Dict[str, Any]] = {}
        for _,row in df.iterrows():
            feed_id = row["feed_id"]
            feeds_lookup[feed_id] = {
                "feed_id"        : feed_id,
                "title"          : row["title"],
                "feed_text"      : row["feed_text"],
                "tags"           : row["tags"],                     
                "language"       : row["language"],
                "created_at"     : row["created_at"],
                "source"         : row["source"],
                "url"            : row["url"],
                "views"          : int(row["views"]),
                "embedding_input": row["embedding_input"]
            }
        return feeds_lookup
# dq = DataQuery()
# dq.get_students()   

### Cloud storage

In [6]:
import json
import numpy as np
import io
from datetime import datetime, timedelta, timezone
from google.cloud import storage

class GoogleCloudStorage:
    def __init__(self,bucket_name):
        self.client = storage.Client()
        try:
            self.bucket = self.client.get_bucket(bucket_name)
            print(f"Bucket exists  : {bucket_name}")
        except Exception:
            self.bucket = self.client.create_bucket(bucket_name, location=location)
            print(f"Bucket created : {bucket_name}")
            
    def blob_exists(self, blob_path) -> bool:
        '''check if object exists'''
        return self.bucket.blob(blob_path).exists()

    ### ---------- Upload folder function ----------- ###
    def upload_json(self,blob_path,json_data):
        '''upload json file to bucket'''
        blob   = self.bucket.blob(blob_path)
        
        blob.upload_from_string(
            json.dumps(json_data,ensure_ascii = False),
            content_type = "application/json"
        )
        print(f"uploaded JSON -> gs://{self.bucket.name}/{blob_path}")

    def upload_text(self, blob_path, text_data):
        '''upload text file to bucket'''
        blob   = self.bucket.blob(blob_path)

        blob.upload_from_string(
            text_data,
            content_type = "text/plain"
        )
        print(f"Uploaded text -> gs://{self.bucket.name}/{blob_path}")

    def upload_npy(self, blob_path, array):
        '''upload embedding vector'''
        buffer = io.BytesIO()
        np.save(buffer, array)
        buffer.seek(0)
        
        blob = self.bucket.blob(blob_path)
        blob.upload_from_file(
            buffer,
            content_type = "application/octet-stream"
        )
        print(f"Uploaded NPY -> gs://{self.bucket.name}/{blob_path}")
        
    ### ---------- Read file function ----------- ###
    def read_json(self, blob_path):
        '''read json file'''
        blob   = self.bucket.blob(blob_path)
        return json.loads(blob.download_as_text())

    def read_text(self, blob_path):
        '''read text file'''
        blob   = self.bucket.blob(blob_path)
        return blob.download_as_text()

    def read_npy(self, blob_path):
        '''read .npy (embedding vector) file'''
        blob   = self.bucket.blob(blob_path)

        buffer = io.BytesIO()
        blob.download_to_file(buffer)
        buffer.seek(0)
        return np.load(buffer)
        
    ### ---------- Creation folder function ----------- ###
    def create_folder(self,folder_path):
        '''Creating folder and sub folder'''
        if not folder_path.endswith("/"):
            folder_path += "/"
        blob = self.bucket.blob(folder_path)
        blob.upload_from_string("")
        print(f"Folder created : gs://{self.bucket}/{folder_path}")
        
    ### ---------- Remove function ----------- ###
    def delete_blob(self, blob_path):
        blob   = self.bucket.blob(blob_path)
        if blob.exists():
            blob.delete()
        print(f"Deleted: gs://{self.bucket_name}/{blob_path}")

    def delete_folder(self, folder_path):
        '''Remove nest blob(file) in folder'''
        if not folder_path.endswith("/"):
            folder_path += "/"
        blobs = self.bucket.list_blobs(prefix=folder_path)
        count = 0
        for blob in blobs:
            blob.delete()
            count += 1
    
        print(f"Deleted {count} objects under gs://{self.bucket_name}/{folder_path}")

    # def delete_by_ttl(self, prefix, ttl: timedelta):
    #     '''Remove folder with setting time
    #     timeformat support
    #     timedelta(
    #         days=...,
    #         seconds=...,
    #         microseconds=...,
    #         milliseconds=...,
    #         minutes=...,
    #         hours=...,
    #         weeks=...
    #     )
    #     '''
    #     now    = datetime.now(timezone.utc)
    #     blob   = self.bucket.blob(blob_path)
    #     deleted = 0
    #     for blob in blobs:
    #         if blob.time_created and now - blob.time_created > ttl:
    #             blob.delete()
    #             deleted += 1
    #     print(f"TTL cleanup deleted {deleted} objects under {prefix}")
        
cgs = GoogleCloudStorage(bucket_name = "hyde-datalake-feeds")

Bucket exists  : hyde-datalake-feeds


### Helper function

In [7]:
def ensure_dir(path: str) -> None:
    """Create directory if it does not exist (idempotent)."""
    os.makedirs(path, exist_ok=True)
    
def _read_hyde_config(cfg: Dict[str, Any]) -> Tuple[int, int, int, bool, str]:
    """
    Read HyDE-related configuration with safe defaults.

    Returns
    -------
    history_threshold:
        Event count threshold for prompt selection
    recent_k:
        Max number of recent feeds used in HistorySummary
    feed_text_max_chars:
        Per-feed text truncation limit
    include_recent_feeds:
        Whether HistorySummary may include feed snippets
    query_embedding_model_name:
        Embedding model for HyDE queries
    """
    hyde_cfg = cfg.get("hyde", {}) if isinstance(cfg, dict) else {}

    history_threshold = int(hyde_cfg.get("history_threshold", 5))
    recent_k = int(hyde_cfg.get("recent_k", 5))
    feed_text_max_chars = int(hyde_cfg.get("feed_text_max_chars", 240))
    include_recent_feeds = bool(hyde_cfg.get("include_recent_feeds", True))

    # Default to same embedding family as feed embeddings
    query_embedding_model_name = str(
        hyde_cfg.get("query_embedding_model_name")
        or cfg.get("embeddings", {}).get("model_name", "")
        or "gemini-embedding-001"
    )

    # Hard safety guards
    history_threshold = max(1, history_threshold)
    recent_k = max(0, min(recent_k, 10))
    feed_text_max_chars = max(0, min(feed_text_max_chars, 2000))

    return (
        history_threshold,
        recent_k,
        feed_text_max_chars,
        include_recent_feeds,
        query_embedding_model_name,
    )
    
def read_jsonl(path: str) -> List[Dict[str, Any]]:
    """
    Deterministic JSONL reader.

    Order is preserved, which is critical for any downstream alignment.
    """
    rows: List[Dict[str, Any]] = []
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception as e:
                raise ValueError(f"Invalid JSONL at line {line_no}: {e}") from e
    return rows

def load_prompts() -> Dict[str, str]:
    """
    Load HyDE prompt templates from parameters/prompts.yaml.

    Expected structure:
      hyde_prompts:
        hyde_a: "..."
        hyde_b: "..."
        hyde_c: "..."
    """
    import yaml

    prompts_path = PROJECT_ROOT / "parameters" / "prompts.yaml"
    with prompts_path.open("r", encoding="utf-8") as f:
        data = yaml.safe_load(f) or {}

    return data.get("hyde_prompts", {}) or {}

# =============================================================================
# Prompt selection and rendering
# =============================================================================
def choose_hyde_prompt_key(num_events: int, history_threshold: int = 5) -> str:
    """
    Select HyDE prompt variant based on interaction volume.

    Rules
    -----
    - num_events >= history_threshold → history-heavy (hyde_b)
    - num_events <= 1               → onboarding / sparse (hyde_c)
    - otherwise                     → mixed (hyde_a)
    """
    if num_events >= history_threshold:
        return "hyde_b"
    if num_events <= 1:
        return "hyde_c"
    return "hyde_a"


def render_prompt(
    template: str,
    preferred_language: str,
    user_context_text: str,
    history_summary_text: Optional[str],
) -> str:
    """
    Render a prompt template using strict placeholder substitution.

    Supported placeholders:
    - {{preferred_language}}
    - {{UserContextText}}
    - {{HistorySummaryText}}

    No templating engine is used on purpose to keep behavior explicit.
    """
    s = template.replace("{{preferred_language}}", preferred_language or "th")
    s = s.replace("{{UserContextText}}", user_context_text or "")
    s = s.replace("{{HistorySummaryText}}", history_summary_text or "")
    return s

# =============================================================================
# HyDE output handling
# =============================================================================
def _extract_hyde_query_texts(hyde_json: Dict[str, Any]) -> List[str]:
    """
    Extract query_text values from HyDE JSON output.

    Expected structure:
      {
        "hyde_queries": [
          {"query_id": "...", "query_text": "...", ...},
          ...
        ]
      }

    Order is preserved and MUST match embedding row order.
    """
    if not isinstance(hyde_json, dict):
        raise ValueError("hyde_output must be a dict")

    items = hyde_json.get("hyde_queries") or []
    if not isinstance(items, list):
        raise ValueError("hyde_output.hyde_queries must be a list")

    out: List[str] = []
    for i, it in enumerate(items):
        if not isinstance(it, dict):
            raise ValueError(f"hyde_output.hyde_queries[{i}] must be an object")
        out.append(str(it.get("query_text") or "").strip())

    return out


def _l2_normalize_rows(x: np.ndarray) -> np.ndarray:
    """
    Row-wise L2 normalization.

    Zero rows are left as zero to avoid NaNs.
    """
    if x.ndim != 2:
        raise ValueError("Expected 2D array for row normalization")

    norms = np.linalg.norm(x, axis=1, keepdims=True)
    norms[norms == 0.0] = 1.0
    return (x / norms).astype(np.float32)


def _atomic_save_npy(path: str, arr: np.ndarray) -> None:
    """
    Best-effort atomic .npy write.

    Writes to a temp file and renames to avoid partial reads.
    """
    tmp = path + ".tmp.npy"
    np.save(tmp, arr)
    os.replace(tmp, path)

# Main

### Load resource

In [8]:
cfg = load_config()
out_dir = cfg["artifacts"]["user_query_bundles_dir"]
verbose = 1
bq = DataQuery()

In [9]:
# students_path = cfg["data"]["students_path"]           # 'data/students.csv'
# students      = pd.read_csv(students_path)
# students
students = bq.get_students() 

/usr/local/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [10]:
students

,student_id,preferred_language,current_status,education_level,education_major,target_roles,skills,interests,onboard_grp,onboard_grp_description
0,stu_p000,en,student,bachelor,electrical engineering,data science,python;sql;statistics,machine learning;career growth,job_hunter,looking to transition into data science role
1,stu_p007,th,newgrad,bachelor,วิศวกรรมอุตสาหการ,Quality Engineer,Statistics:L1;Documentation:L1,เตรียมสัมภาษณ์;หางาน,Job_Hunter,เพิ่งจบและหางานสายคุณภาพ/กระบวนการ
2,stu_p006,th,student2yr,bachelor,ชีววิทยา,Biotechnology Intern,Biology Fundamentals:L1;Lab Skills:unknown,ฝึกงาน;ทำพอร์ตสาย Bio;สมัครทุน,Job_Hunter,อยากได้ฝึกงานสายชีวภาพและอยากเตรียมพอร์ต
3,stu_p001,th,student3yr,bachelor,วิทยาการคอมพิวเตอร์,Data Analyst,Python:L2;SQL:L2,ทำพอร์ต;ฝึกสัมภาษณ์,Job_Hunter,เตรียมฝึกงานสายข้อมูล
4,stu_p009,en,student3yr,bachelor,Computer Science,Data Analyst,SQL:L1;Python:L1,portfolio;internship;interview,Job_Hunter,Looking for internship and building a data por...
5,stu_p003,th,student4+yr,bachelor,สถิติ,Data Analyst,Statistics:L2;Excel:L2;Basic SQL:L1,เตรียมเรซูเม่;ฝึกสัมภาษณ์;ทำโปรเจกต์,Job_Hunter,ใกล้จบและอยากสมัครงาน Data Analyst
6,stu_p004,th,student4+yr,bachelor,บริหารธุรกิจ,Business Analyst,Excel:L2;Presentation:L1,หางาน;ทำเรซูเม่;ฝึกสัมภาษณ์,Job_Hunter,ใกล้จบและกำลังหางานสายวิเคราะห์ธุรกิจ
7,stu_p005,th,student1yr,bachelor,วิทยาการคอมพิวเตอร์,Undecided,Python:L1,สำรวจสายอาชีพ;เรียนรู้พื้นฐาน,Learner,ยังไม่ชัดเจน เป้าหมายคือสำรวจสายงานและสร้างพื้...
8,stu_p002,th,student2yr,bachelor,เทคโนโลยีชีวภาพ,Biotechnology Researcher|Lab Scientist,Lab Skills:L1;Molecular Biology:L1,ทำวิจัย;ฝึกงานแลบ;สมัครทุน,Learner,อยากทำวิจัยและเตรียมตัวเข้าฝึกงานสายชีวภาพ
9,stu_p010,th,student3yr,bachelor,ความสัมพันธ์ระหว่างประเทศ,Policy Analyst,Writing:L1;Research:L1,สมัครทุน;ข่าวมหาวิทยาลัย;เส้นทางอาชีพ,Learner,สนใจทุนและเส้นทางอาชีพด้านนโยบาย (harder match)


In [11]:
# interactions_path = cfg["data"]["interactions_path"]   # 'data/interactions.csv'
# interactions = pd.read_csv(interactions_path)
# interactions
interactions = bq.get_interactions() 

/usr/local/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [12]:
# feeds_path = cfg["data"]["feeds_path"]                 # 'data/feeds.jsonl'
# feeds_lookup: Dict[str, Dict[str, Any]] = {}
# if os.path.exists(feeds_path):
#     feeds = read_jsonl(feeds_path)
#     feeds_lookup = {
#         str(f.get("feed_id")): f
#         for f in feeds
#         if isinstance(f, dict) and f.get("feed_id") is not None
#     }
feeds_lookup = bq.get_user_events_json()

### Read HyDE-related configuration once

In [13]:
(history_threshold,recent_k,feed_text_max_chars,include_recent_feeds,query_embedding_model_name) = _read_hyde_config(cfg)
expected_dim = int(cfg.get("embeddings", {}).get("dim", 0) or 0)

In [14]:
prompts = load_prompts()
if not prompts:
    raise ValueError("hyde_prompts missing from parameters/prompts.yaml")
client = build_llm_client_from_yaml(
    parameters_path=str(PROJECT_ROOT / "parameters" / "parameters.yaml"),
    credentials_path=str(PROJECT_ROOT / "parameters" / "credentials.yaml"),
)
query_embedder = GoogleEmbeddingModel(
    model_name=query_embedding_model_name,
    credentials_path=str(PROJECT_ROOT / "parameters" / "credentials.yaml"),
)
now_iso = datetime.now(timezone.utc).replace(microsecond=0).isoformat()

In [15]:
query_embedder

GoogleEmbeddingModel(model_name='gemini-embedding-001', credentials_path='/code/src/parameters/credentials.yaml', output_dim=768, uniqueness_guard_enabled=True, uniqueness_guard_min_unique_ratio=0.85, uniqueness_guard_round_decimals=8, _client=None)

In [16]:
cgs = GoogleCloudStorage(bucket_name = "hyde-datalake-feeds")

Bucket exists  : hyde-datalake-feeds


In [17]:
verbose = 0

In [18]:
# ------------------------------------------------------------------
# Generate one cached bundle per student
# ------------------------------------------------------------------
rows = []
logger = get_logger("pipeline_1_user_hyde")
for _, row in students.iterrows():
    student_row = row.to_dict()     # convert pd -> dict for each row
    student_id  = str(student_row.get("student_id","")).strip()
    # if student_id != "stu_p000":
    #     continue
    if not student_id or student_id.lower() == "nan":
        raise ValueError(f"Invalid student_id in students.csv: {student_row!r}")
    
    user_ctx = build_user_context(student_row)
    pref_lang = user_ctx.user_context_json.get("preferred_language","th")

    user_events = interactions[interactions["user_id"] == student_id]   # <- user event from interaction.csv
    num_events  = int(len(user_events))

    history_summary_text : Optional[str] = None
    #** Crate by combe data for each person student **#
    if num_events > 0:
        history_summary_text = build_history_summary(
            user_events,
            preferred_language   = pref_lang,
            include_recent_feeds = include_recent_feeds,
            recent_k             = recent_k,
            feeds_lookup         = feeds_lookup or None,
            feed_text_max_chars  = feed_text_max_chars,
        )
    
    prompt_key = choose_hyde_prompt_key(num_events,history_threshold)
    template = prompts.get(prompt_key)
    if not template:
        raise ValueError(f"Missing prompt '{prompt_key}' in pormpts.yaml")
    prompt = render_prompt(
            template=template,
            preferred_language=pref_lang,
            user_context_text=user_ctx.user_context_text,
            history_summary_text=history_summary_text,
        )
    # ------------------------------------------------------------------
    # LLM call (JSON-only)
    # ------------------------------------------------------------------
    hyde_json = client.generate_json(prompt)
    # ------------------------------------------------------------------
    # Embed HyDE queries for fast serving
    # ------------------------------------------------------------------
    hyde_query_texts = _extract_hyde_query_texts(hyde_json)

    if hyde_query_texts:
        emb = query_embedder.embed_documents(hyde_query_texts)
        print("#"*100)
        print(f"emb.shape ->\n{emb.shape}")
        emb = np.asarray(emb, dtype = np.float32)
        if emb.ndim != 2:
            raise ValueError(f"Invalid embedding shape {emb.shape}")
        emb = _l2_normalize_rows(emb)
        if expected_dim and emb.shape[1] != expected_dim:
            raise ValueError(
                f"Embedding dim mismatch for student = {student_id}:"
                f"got {emb.shape[1]} expected {expected_dim}"
            )
        dim = int(emb.shape[1])
    else:
        dim = expected_dim or 0
        emb = np.zeros((0,dim), dtype=np.float32)

    emb_filename = f"{student_id}_hyde_q_emb.npy"
    emb_path     = os.path.join(out_dir, emb_filename)
    _atomic_save_npy(emb_path, emb)
    # ---------------------------------------------------
    # Persist cached bundle for online serving
    # ---------------------------------------------------
    bundle: Dict[str, Any] = {
        "bundle_version"        : "v2_hyde_embedded_queries",
        "student_id"            : student_id,
        "generated_at"          : now_iso,
        "prompt_key"            : prompt_key,
        "preferred_language"    : pref_lang,
        "num_events"            : num_events,
        "user_context_json"     : user_ctx.user_context_json,
        "user_context_text"     : user_ctx.user_context_text,
        "history_summary_text"  : history_summary_text,
        "hyde_output"           : hyde_json,
        "hyde_query_embeddings" : {
            "path"        : emb_filename,
            "model"       : query_embedding_model_name,
            "dim"         : dim,
            "dtype"       : "float32",
            "num_queries" : int(len(hyde_query_texts)),
            "normalized"  : True,
        },
    }

    out_path = os.path.join(out_dir, f"{student_id}.json")
    with open(out_path,"w",encoding="utf-8") as f:
        json.dump(bundle,f,ensure_ascii=False,indent=2)
    logger.info(
        "wrote HyDE bundle student_id=%s events=%d prompt=%s",
        student_id,
        num_events,
        prompt_key,
    )
    if verbose > 0:
        print(_)
        print(f"student_row -> \n {student_row}")
        print(f"user_ctx -> \n {user_ctx}")
        print(f"user_events -> \n {user_events}")
        print(f"promt_key -> {prompt_key}")
        print(f"hyde_query_texts->{hyde_query_texts}")
        print(f"emb -> \n{emb}")
        print(f"emb_path ->\n{emb_path}")
        print(f"bundle->\n{bundle}")
        print("#"*100)
    # for vec in emb:  # emb.shape = (N, D)
    #     rows.append({
    #         "user_id": student_row["student_id"],
    #         "created_at": datetime.now(timezone.utc).isoformat(),
    #         # "embedding": vec.tolist()
    #         "embedding": "x"
    
    #     })
    # Ingest to GCS
    cgs.create_folder(
        folder_path = f"{student_id}/embedding/"
    )
    cgs.create_folder(
        folder_path = f"{student_id}/metadata/"
    )
    metadata = {
        "student_id":student_id,
        "current_status":student_row['current_status'],
        "education_level":student_row['education_level'],
        "education_major":student_row['education_major'],
        "target_roles":student_row['target_roles'],
        "timezone":cfg["app"]["timezone"],
        "model_name":cfg["llm"]["model_name"],
        "max_output_tokens":cfg["llm"]["max_output_tokens"],
        "feed_text_max_chars":cfg["hyde"]["feed_text_max_chars"],
        "temperature":cfg["llm"]["temperature"]
    }
    print(metadata)
    cgs.upload_json(
        blob_path   = f"{student_id}/metadata/metadata.json",
        json_data   = metadata
    )
    cgs.upload_npy(
        blob_path   = f"{student_id}/embedding/embedding01.npy",
        array       = emb[0]
    )
    cgs.upload_npy(
        blob_path   = f"{student_id}/embedding/embedding02.npy",
        array       = emb[1]
    )
    cgs.upload_npy(
        blob_path   = f"{student_id}/embedding/embedding03.npy",
        array       = emb[2]
    )
    cgs.upload_npy(
        blob_path   = f"{student_id}/embedding/embedding04.npy",
        array       = emb[3]
    )
    cgs.upload_npy(
        blob_path   = f"{student_id}/embedding/embedding05.npy",
        array       = emb[4]
    )
    
    # break

2026-02-04T02:42:07Z | INFO | functions.utils.llm_client | LLM call done | attempt=1 | latency=7.734s | in_tokens=782 | out_tokens=307 | model=gemini-2.5-flash | status=ok
2026-02-04T02:42:10Z | INFO | pipeline_1_user_hyde | wrote HyDE bundle student_id=stu_p000 events=4 prompt=hyde_a


####################################################################################################
emb.shape ->
(5, 768)
Folder created : gs://<Bucket: hyde-datalake-feeds>/stu_p000/embedding/
Folder created : gs://<Bucket: hyde-datalake-feeds>/stu_p000/metadata/
{'student_id': 'stu_p000', 'current_status': 'student', 'education_level': 'bachelor', 'education_major': 'electrical engineering', 'target_roles': 'data science', 'timezone': 'UTC', 'model_name': 'gemini-2.5-flash', 'max_output_tokens': 2048, 'feed_text_max_chars': 240, 'temperature': 0.2}
uploaded JSON -> gs://hyde-datalake-feeds/stu_p000/metadata/metadata.json
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p000/embedding/embedding01.npy
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p000/embedding/embedding02.npy
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p000/embedding/embedding03.npy
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p000/embedding/embedding04.npy
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p000/embedding/embe

2026-02-04T02:42:20Z | INFO | functions.utils.llm_client | LLM call done | attempt=1 | latency=8.782s | in_tokens=816 | out_tokens=338 | model=gemini-2.5-flash | status=ok
2026-02-04T02:42:22Z | INFO | pipeline_1_user_hyde | wrote HyDE bundle student_id=stu_p007 events=4 prompt=hyde_a


####################################################################################################
emb.shape ->
(5, 768)
Folder created : gs://<Bucket: hyde-datalake-feeds>/stu_p007/embedding/
Folder created : gs://<Bucket: hyde-datalake-feeds>/stu_p007/metadata/
{'student_id': 'stu_p007', 'current_status': 'newgrad', 'education_level': 'bachelor', 'education_major': 'วิศวกรรมอุตสาหการ', 'target_roles': 'Quality Engineer', 'timezone': 'UTC', 'model_name': 'gemini-2.5-flash', 'max_output_tokens': 2048, 'feed_text_max_chars': 240, 'temperature': 0.2}
uploaded JSON -> gs://hyde-datalake-feeds/stu_p007/metadata/metadata.json
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p007/embedding/embedding01.npy
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p007/embedding/embedding02.npy
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p007/embedding/embedding03.npy
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p007/embedding/embedding04.npy
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p007/embedding/embed

2026-02-04T02:42:33Z | WARNING | functions.utils.llm_client | LLM JSON parse failed (will retry) | attempt=1 | latency=9.163s | in_tokens=965 | out_tokens=312 | model=gemini-2.5-flash | first_200={\n  "output_language": "th",\n  "hyde_queries": [\n    {\n      "query_id": "Q1",\n      "query_text": "จัดพอร์ตสมัครฝึกงานสายชีวภาพสำหรับนักศึกษาชีววิทยา",\n      "weight": 1.0,\n      "intent_label": "his
2026-02-04T02:42:33Z | WARNING | functions.utils.llm_client | Retrying functions.utils.llm_client.GeminiJsonClient.generate_json.<locals>._call_once in 1.0 seconds as it raised ValueError: LLM returned malformed or non-extractable JSON.
2026-02-04T02:42:42Z | INFO | functions.utils.llm_client | LLM call done | attempt=2 | latency=7.732s | in_tokens=965 | out_tokens=330 | model=gemini-2.5-flash | status=ok
2026-02-04T02:42:44Z | INFO | pipeline_1_user_hyde | wrote HyDE bundle student_id=stu_p006 events=6 prompt=hyde_b


...

####################################################################################################
emb.shape ->
(5, 768)
Folder created : gs://<Bucket: hyde-datalake-feeds>/stu_p002/embedding/
Folder created : gs://<Bucket: hyde-datalake-feeds>/stu_p002/metadata/
{'student_id': 'stu_p002', 'current_status': 'student2yr', 'education_level': 'bachelor', 'education_major': 'เทคโนโลยีชีวภาพ', 'target_roles': 'Biotechnology Researcher|Lab Scientist', 'timezone': 'UTC', 'model_name': 'gemini-2.5-flash', 'max_output_tokens': 2048, 'feed_text_max_chars': 240, 'temperature': 0.2}
uploaded JSON -> gs://hyde-datalake-feeds/stu_p002/metadata/metadata.json
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p002/embedding/embedding01.npy
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p002/embedding/embedding02.npy
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p002/embedding/embedding03.npy
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p002/embedding/embedding04.npy
Uploaded NPY -> gs://hyde-datalake-feeds/s

2026-02-04T02:44:07Z | INFO | functions.utils.llm_client | LLM call done | attempt=1 | latency=7.450s | in_tokens=776 | out_tokens=354 | model=gemini-2.5-flash | status=ok
2026-02-04T02:44:09Z | INFO | pipeline_1_user_hyde | wrote HyDE bundle student_id=stu_p010 events=3 prompt=hyde_a


####################################################################################################
emb.shape ->
(5, 768)
Folder created : gs://<Bucket: hyde-datalake-feeds>/stu_p010/embedding/
Folder created : gs://<Bucket: hyde-datalake-feeds>/stu_p010/metadata/
{'student_id': 'stu_p010', 'current_status': 'student3yr', 'education_level': 'bachelor', 'education_major': 'ความสัมพันธ์ระหว่างประเทศ', 'target_roles': 'Policy Analyst', 'timezone': 'UTC', 'model_name': 'gemini-2.5-flash', 'max_output_tokens': 2048, 'feed_text_max_chars': 240, 'temperature': 0.2}
uploaded JSON -> gs://hyde-datalake-feeds/stu_p010/metadata/metadata.json
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p010/embedding/embedding01.npy
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p010/embedding/embedding02.npy
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p010/embedding/embedding03.npy
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p010/embedding/embedding04.npy
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p010/embedd

2026-02-04T02:44:18Z | INFO | functions.utils.llm_client | LLM call done | attempt=1 | latency=7.224s | in_tokens=910 | out_tokens=328 | model=gemini-2.5-flash | status=ok
2026-02-04T02:44:20Z | INFO | pipeline_1_user_hyde | wrote HyDE bundle student_id=stu_p008 events=5 prompt=hyde_b


####################################################################################################
emb.shape ->
(5, 768)
Folder created : gs://<Bucket: hyde-datalake-feeds>/stu_p008/embedding/
Folder created : gs://<Bucket: hyde-datalake-feeds>/stu_p008/metadata/
{'student_id': 'stu_p008', 'current_status': 'student0yr', 'education_level': 'bachelor', 'education_major': 'นิเทศศาสตร์', 'target_roles': 'Undecided', 'timezone': 'UTC', 'model_name': 'gemini-2.5-flash', 'max_output_tokens': 2048, 'feed_text_max_chars': 240, 'temperature': 0.2}
uploaded JSON -> gs://hyde-datalake-feeds/stu_p008/metadata/metadata.json
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p008/embedding/embedding01.npy
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p008/embedding/embedding02.npy
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p008/embedding/embedding03.npy
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p008/embedding/embedding04.npy
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p008/embedding/embedding05.npy

<hr>

In [19]:
metadata = cgs.read_json("stu_p010/metadata/metadata.json")
metadata

{'student_id': 'stu_p010',
 'current_status': 'student3yr',
 'education_level': 'bachelor',
 'education_major': 'ความสัมพันธ์ระหว่างประเทศ',
 'target_roles': 'Policy Analyst',
 'timezone': 'UTC',
 'model_name': 'gemini-2.5-flash',
 'max_output_tokens': 2048,
 'feed_text_max_chars': 240,
 'temperature': 0.2}

In [20]:
emb1  = cgs.read_npy("stu_p002/embedding/embedding01.npy")
emb1

array([ 2.02630796e-02, -1.00783184e-02, -1.17169265e-02, -9.33991298e-02,
        1.11293215e-02,  1.67154837e-02, -7.31987599e-03,  9.11151804e-03,
        1.78892948e-02,  2.02817731e-02, -2.56754365e-02, -7.72794848e-03,
        3.76069127e-03, -6.50635734e-03,  1.63597271e-01, -4.19840291e-02,
        8.25960841e-03, -1.43025862e-02, -7.30392570e-03,  1.08699603e-02,
       -1.57709792e-03,  3.23276669e-02, -1.86694153e-02, -6.71615005e-02,
       -3.51384655e-02, -3.36544402e-02,  3.01890541e-02,  5.43543436e-02,
        3.81472595e-02, -2.95914374e-02,  2.14089490e-02,  3.54561880e-02,
        3.91543321e-02,  1.75003689e-02, -2.09581498e-02, -4.28141124e-04,
       -4.91309678e-03, -1.82263050e-02,  3.22161056e-03, -4.60411534e-02,
       -2.86514405e-02,  2.97178309e-02, -4.47208174e-02, -7.20117614e-03,
        1.14270095e-02, -1.01002371e-02,  2.69028563e-02, -4.65039909e-02,
       -2.50538122e-02,  1.25069590e-02,  2.12856121e-02,  7.47426301e-02,
       -3.12850066e-02, -

In [23]:
cgs = GoogleCloudStorage(bucket_name = "hyde-datalake-feeds")
student_id = "stu_p002"
metadata = cgs.read_json(f"{student_id}/metadata/metadata.json")
emb1     = cgs.read_npy(f"{student_id}/embedding/embedding01.npy")
emb2     = cgs.read_npy(f"{student_id}/embedding/embedding02.npy")
emb3     = cgs.read_npy(f"{student_id}/embedding/embedding03.npy")
emb4     = cgs.read_npy(f"{student_id}/embedding/embedding04.npy")
emb5     = cgs.read_npy(f"{student_id}/embedding/embedding05.npy")
metadata

{'student_id': 'stu_p002',
 'current_status': 'student2yr',
 'education_level': 'bachelor',
 'education_major': 'เทคโนโลยีชีวภาพ',
 'target_roles': 'Biotechnology Researcher|Lab Scientist',
 'timezone': 'UTC',
 'model_name': 'gemini-2.5-flash',
 'max_output_tokens': 2048,
 'feed_text_max_chars': 240,
 'temperature': 0.2}